# 08 – Validation sheet, agreement script and the dataset's own labels

1. Build the 50-chunk validation sheet (25 chatbot answers, 25 news chunks) so that a human can label which frame each text emphasises most, **without seeing the dictionary's answer**, and check that the files open cleanly.
2. The agreement script (`agreement.py`): dictionary against human labels, with per-frame precision, recall and Cohen's kappa. Tested here on dummy labels.
3. Compare the dictionary's news scores with the **dataset's own frame labels** (`brewcoua/genai-gdelt-framing`), and list disagreements.

**Labelling is left to the team.** Nothing in this notebook produces human labels; the dummy labels in section 3 exist only to test the script and are never saved.

In [1]:
import json, re, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from agreement import agreement_report, cohen_kappa, per_class_report, confusion
from sklearn.metrics import cohen_kappa_score

CFG = json.load(open("config.json"))
OUT = Path(CFG["analysis_dir"]); FR = Path("dataset/frames"); DATA = Path("data")
SEED = 42
FRAMES = list(json.load(open(CFG["lexicon_json"]))["frames"]); HC = [f"hits_{f}" for f in FRAMES]
LABELS = FRAMES + ["none"]

## 1. The validation sheet
- **Stratified:** chatbot answers 13 Llama + 12 Qwen; news chunks 7 + 6 + 6 + 6 across the four regions. Random within each stratum (seeded).
- **No leaks.** The sheet the labeller gets has only an opaque `label_id`, the text and an empty `human_frame` column: no prompt id (it carries the wording), no source, no lexicon label, no hit counts. A separate **key** file maps `label_id` back to the source and holds the dictionary's top frame. The dictionary's top frame is the frame with the most hits; ties go to the frame listed first in the lexicon; `none` when the text has no hits.
- **Instruction row:** `label_id = 0` is an instruction row (ignored by the scoring script), so the sheet is self-explaining when opened in a spreadsheet. Full definitions are in `dataset/frames/validation_instructions.md`.

In [2]:
def head_words(text, n=300):
    for k, m in enumerate(re.finditer(r"\S+", text)):
        if k == n - 1:
            return text[:m.end()]
    return text

ans = pd.read_json(CFG["answers_jsonl"], lines=True); ans = ans[ans["chatbot"].isin(["llama3.1:8b", "qwen3.5:9b"])].reset_index(drop=True)
sc_a = pd.read_csv(OUT / "frame_scores_chatbot.csv"); assert (sc_a["prompt_id"] == ans["prompt_id"]).all()
news = pd.read_csv(CFG["news_corpus_csv"], keep_default_na=False); sc_n = pd.read_csv(OUT / "frame_scores_news.csv"); assert (sc_n["doc_id"] == news["doc_id"]).all()

def top_frame(row):
    h = row[HC].to_numpy()
    return "none" if h.sum() == 0 else FRAMES[int(np.argmax(h))]          # argmax: ties go to the first frame in the lexicon order

def pick(df, by, sizes):
    return pd.concat([df[df[by] == k].sample(n, random_state=SEED) for k, n in sizes.items()])

a_pick = pick(sc_a.assign(i=range(len(sc_a))), "chatbot", {"llama3.1:8b": 13, "qwen3.5:9b": 12})
n_pick = pick(sc_n.assign(i=range(len(sc_n))), "region", {"US": 7, "UK": 6, "EU": 6, "Other": 6})
rows = [{"source": "chatbot", "ref": f"{ans.loc[r.i, 'prompt_id']}|r{ans.loc[r.i, 'replicate']}|{ans.loc[r.i, 'chatbot']}", "text": head_words(ans.loc[r.i, "answer"]),
         "lexicon_top_frame": top_frame(r), "lexicon_hits": int(r.hits_total)} for _, r in a_pick.iterrows()]
rows += [{"source": "news", "ref": news.loc[r.i, "doc_id"], "text": head_words(news.loc[r.i, "text"]), "lexicon_top_frame": top_frame(r), "lexicon_hits": int(r.hits_total)}
         for _, r in n_pick.iterrows()]
sheet = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
sheet.insert(0, "label_id", range(1, len(sheet) + 1))

INSTR = ("INSTRUCTIONS (row 0, ignore when scoring): read each text and put in human_frame the ONE frame it emphasises most: "
         + ", ".join(FRAMES) + ", or none (no clear frame). Judge emphasis, not keywords. Do not discuss labels with the other labellers. Definitions: dataset/frames/validation_instructions.md")
blank = pd.concat([pd.DataFrame([{"label_id": 0, "text": INSTR, "human_frame": ""}]), sheet[["label_id", "text"]].assign(human_frame="")], ignore_index=True)
key = sheet[["label_id", "source", "ref", "lexicon_top_frame", "lexicon_hits"]]
blank.to_csv(DATA / "validation_sheet_blank.csv", index=False); key.to_csv(DATA / "validation_sheet_key.csv", index=False)
print(len(sheet), "chunks:", sheet["source"].value_counts().to_dict(), "| dictionary top frame in the key:", key["lexicon_top_frame"].value_counts().to_dict())

50 chunks: {'chatbot': 25, 'news': 25} | dictionary top frame in the key: {'economy_labour': 15, 'regulation_governance': 11, 'risk_safety': 10, 'innovation_opportunity': 9, 'rights_privacy': 3, 'misinformation_integrity': 1, 'none': 1}


In [3]:
defs = '''# Validation sheet: how to label

For each text choose the ONE frame it emphasises most (what most of the text is about), or `none`.
Read the text, do not search for keywords. Use only the text in the sheet. Fill in the `human_frame` column with the exact frame name.

| frame | choose it when the text mainly talks about |
|---|---|
| innovation_opportunity | benefits, breakthroughs, progress, new products, opportunity, startups |
| risk_safety | harms, dangers, accidents, misuse, safety testing, threats, children harmed |
| regulation_governance | regulators, oversight, enforcement, compliance, treaties, standards, legal responsibility, institutions |
| rights_privacy | privacy, personal data, surveillance, discrimination, civil liberties, copyright |
| economy_labour | jobs, workers, automation, markets, competition between firms or countries, costs, trade |
| misinformation_integrity | false information, deepfakes, hallucinations, manipulation, election integrity, authenticity |
| none | none of these clearly dominates (for example a product review or a personal essay) |

Rules: one label per text; if two frames seem equal choose the one that appears first in the text; do not look at the key file until everyone has finished.
'''
(FR / "validation_instructions.md").write_text(defs)
print("written: dataset/frames/validation_instructions.md")

written: dataset/frames/validation_instructions.md


### Does the sheet open cleanly?

In [4]:
b = pd.read_csv(DATA / "validation_sheet_blank.csv", keep_default_na=False); k = pd.read_csv(DATA / "validation_sheet_key.csv", keep_default_na=False)
checks = {
 "blank sheet has exactly the columns label_id, text, human_frame": list(b.columns) == ["label_id", "text", "human_frame"],
 "instruction row present (label_id 0) and the label column is empty everywhere": b.iloc[0]["label_id"] == 0 and b["human_frame"].eq("").all(),
 "50 data rows + 1 instruction row": len(b) == 51 and b["label_id"].iloc[1:].tolist() == list(range(1, 51)),
 "label ids unique": b["label_id"].is_unique,
 "no empty texts; texts are at most 300 words": b["text"].str.len().gt(0).all() and b["text"].iloc[1:].map(lambda t: len(t.split())).le(300).all(),
 "blank sheet leaks nothing: no source, prompt id, model name or lexicon column": not set(b.columns) & {"source", "ref", "lexicon_top_frame", "lexicon_hits", "prompt_id", "chatbot"},
 "no prompt id, model name or doc id inside the texts": not b["text"].iloc[1:].str.contains(r"Q\d\d_|llama3|qwen3|news_\d{4}", regex=True).any(),
 "key is a separate file with the mapping": list(k.columns) == ["label_id", "source", "ref", "lexicon_top_frame", "lexicon_hits"] and k["label_id"].tolist() == list(range(1, 51)),
 "key frame labels are valid": k["lexicon_top_frame"].isin(LABELS).all(),
}
for name, ok in checks.items(): print("OK  " if ok else "FAIL", name)
assert all(checks.values())
b.drop(columns="text").head(4)      # text column not shown: article text is not kept in committed notebook outputs

OK   blank sheet has exactly the columns label_id, text, human_frame
OK   instruction row present (label_id 0) and the label column is empty everywhere
OK   50 data rows + 1 instruction row
OK   label ids unique
OK   no empty texts; texts are at most 300 words
OK   blank sheet leaks nothing: no source, prompt id, model name or lexicon column
OK   no prompt id, model name or doc id inside the texts
OK   key is a separate file with the mapping
OK   key frame labels are valid


,label_id,human_frame
0,0,
1,1,
2,2,
3,3,


## 2. The agreement script, tested on dummy labels
`agreement.py` holds `agreement_report` (accuracy, Cohen's kappa, per-frame precision/recall/F1, confusion matrix). The checks use cases whose answers can be worked out by hand and compare kappa with scikit-learn's.

In [5]:
# 1) identical labels
y = np.array(["a", "b", "c", "a", "b", "c"]); r = agreement_report(y, y, ["a", "b", "c"])
assert r["accuracy"] == 1 and abs(r["kappa"] - 1) < 1e-12 and (r["per_class"][["precision", "recall"]] == 1).all().all()

# 2) worked example: 10 items, 8 agree, each label used 5 times by both -> p_o = 0.8, p_e = 0.5, kappa = 0.6; precision = recall = 0.8
h = np.array(["A"] * 5 + ["B"] * 5); d = np.array(["A"] * 4 + ["B"] * 5 + ["A"])
r = agreement_report(h, d, ["A", "B"]); assert abs(r["accuracy"] - 0.8) < 1e-12 and abs(r["kappa"] - 0.6) < 1e-12
assert np.allclose(r["per_class"][["precision", "recall"]].to_numpy(), 0.8)

# 3) worked example with three labels and unequal use: checked by hand against p_o and p_e
h = np.array(["x"] * 6 + ["y"] * 3 + ["z"] * 1); d = np.array(["x"] * 5 + ["y"] * 4 + ["z"] * 1)   # one x was called y
p_o = 9 / 10; p_e = (6 / 10) * (5 / 10) + (3 / 10) * (4 / 10) + (1 / 10) * (1 / 10)
assert abs(cohen_kappa(h, d) - (p_o - p_e) / (1 - p_e)) < 1e-12

# 4) agrees with scikit-learn on random labels; independent random labels give kappa near 0
rng = np.random.default_rng(SEED); a = rng.choice(LABELS, 5000); c = rng.choice(LABELS, 5000)
assert abs(cohen_kappa(a, c, LABELS) - cohen_kappa_score(a, c)) < 1e-12 and abs(cohen_kappa(a, c, LABELS)) < 0.05
# 5) a label the checked side never uses -> recall 0, precision undefined (NaN), no crash
r = agreement_report(["a", "b"], ["a", "a"], ["a", "b"]); assert r["per_class"].loc[1, "recall"] == 0 and np.isnan(r["per_class"].loc[1, "precision"])
print("agreement script: 5 tests passed (identical, two worked examples, scikit-learn parity and chance level, empty class)")

agreement script: 5 tests passed (identical, two worked examples, scikit-learn parity and chance level, empty class)


In [6]:
def score_sheet(human_csv, key_csv=DATA / "validation_sheet_key.csv", save=True, tag="validation"):
    """Dictionary vs human labels. Reads the labelled sheet and the key, ignores the instruction row, checks every label is valid."""
    h = pd.read_csv(human_csv, keep_default_na=False); h = h[h["label_id"] != 0]
    skipped = int((h["human_frame"] == "skip").sum()); h = h[h["human_frame"] != "skip"]          # 'skip' = the labeller skipped a distressing text
    bad = sorted(set(h["human_frame"]) - set(LABELS))
    if bad or (h["human_frame"] == "").any():
        raise ValueError(f"empty or invalid labels: {bad}")
    m = h.merge(pd.read_csv(key_csv, keep_default_na=False), on="label_id")
    rep = agreement_report(m["human_frame"], m["lexicon_top_frame"], LABELS); rep["skipped"] = skipped
    by_source = {s: agreement_report(g["human_frame"], g["lexicon_top_frame"], LABELS) for s, g in m.groupby("source")}
    if save:
        rep["per_class"].round(3).to_csv(OUT / f"{tag}_per_frame.csv", index=False); rep["confusion"].to_csv(OUT / f"{tag}_confusion.csv")
        pd.DataFrame([{"scope": "all", "n": rep["n"], "skipped": skipped, "accuracy": rep["accuracy"], "kappa": rep["kappa"]}] +
                     [{"scope": s, "n": r["n"], "accuracy": r["accuracy"], "kappa": r["kappa"]} for s, r in by_source.items()]).round(3).to_csv(OUT / f"{tag}_summary.csv", index=False)
    return rep, by_source

# end-to-end test with DUMMY human labels (a fake labeller who agrees with the dictionary 70% of the time); written to a temp file, never saved in the repo
rng = np.random.default_rng(SEED); kk = pd.read_csv(DATA / "validation_sheet_key.csv", keep_default_na=False)
fake = [t if rng.random() < 0.7 else rng.choice(LABELS) for t in kk["lexicon_top_frame"]]
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "dummy.csv"; pd.concat([blank.iloc[:1], blank.iloc[1:].assign(human_frame=fake)]).to_csv(p, index=False)
    rep, by_source = score_sheet(p, save=False)
with tempfile.TemporaryDirectory() as td:               # a skipped text is dropped and counted, not an error
    p2 = Path(td) / "dummy_skip.csv"; sk = fake[:]; sk[0] = "skip"; sk[7] = "skip"
    pd.concat([blank.iloc[:1], blank.iloc[1:].assign(human_frame=sk)]).to_csv(p2, index=False)
    r_skip, _ = score_sheet(p2, save=False); assert r_skip["n"] == 48 and r_skip["skipped"] == 2
print("dummy labels: accuracy", round(rep["accuracy"], 2), "| kappa", round(rep["kappa"], 2), "| n", rep["n"], "(not saved; the pipeline works end to end)")
rep["per_class"].round(2)

dummy labels: accuracy 0.7 | kappa 0.63 | n 50 (not saved; the pipeline works end to end)


,label,n_reference,n_checked,true_positive,precision,recall,f1
0,innovation_opportunity,8,9,7,0.78,0.88,0.82
1,risk_safety,6,10,5,0.50,0.83,0.62
2,regulation_governance,13,11,10,0.91,0.77,0.83
3,rights_privacy,4,3,2,0.67,0.50,0.57
4,economy_labour,12,15,10,0.67,0.83,0.74
5,misinformation_integrity,4,1,0,0.00,0.00,NaN
6,none,3,1,1,1.00,0.33,0.50


**To score real labels:** save the labelled sheet (for example `data/validation_sheet_labelled_A.csv`), then run `score_sheet("data/validation_sheet_labelled_A.csv")`. It stops with an error on empty or misspelled labels, and writes `validation_per_frame.csv`, `validation_confusion.csv` and `validation_summary.csv` to `dataset/analysis/`. Two labellers can be compared with `agreement_report(labels_A, labels_B, LABELS)`.

## 3. Dictionary vs the dataset's own frame labels
**These are two different instruments, so disagreement is expected.** The dataset assigns frames from GDELT's *entity-name and quotation fields* (not the article body), keyword match plus an embedding confirmation: a frame is *confirmed* when `kw_* == 1` and `emb_* > 0`; `dominant_frame` is the confirmed frame with the highest normalised keyword count, null if none is confirmed. Our dictionary reads the first 300 words of the article body. Agreement here says how far the two instruments move together; it does not make either one the truth.

Compared for the 420 corpus articles, matched on the normalised URL: (a) per frame, our "frame present" (at least 1 hit) against the dataset's "frame confirmed"; (b) our top frame against the dataset's `dominant_frame` where the dataset has one.

In [7]:
import pyarrow.parquet as pq
ds_cols = [c for c in pq.ParquetFile(CFG["parquet"]).schema_arrow.names if c.startswith(("kw_", "emb_"))]
ds = pd.read_parquet(CFG["parquet"], columns=["document_id", "dominant_frame"] + ds_cols)
ds["url"] = ds["document_id"].str.split("?").str[0].str.rstrip("/"); ds = ds.drop_duplicates("url")
MAP = {"economic_competition_labour": "economy_labour"}
ds["dominant_frame"] = ds["dominant_frame"].replace(MAP)
for f in FRAMES:
    kw = "kw_economic_competition_labour" if f == "economy_labour" else f"kw_{f}"; em = "emb_economic_competition_labour" if f == "economy_labour" else f"emb_{f}"
    ds[f"confirmed_{f}"] = ((ds[kw] == 1) & (ds[em] > 0)).astype(int)
cmp_ = news[["doc_id", "url", "title", "region"]].merge(ds[["url", "dominant_frame"] + [f"confirmed_{f}" for f in FRAMES]], on="url", how="left")
cmp_[HC] = sc_n[HC].to_numpy(); cmp_["hits_total"] = sc_n["hits_total"].to_numpy()
cmp_["our_top_frame"] = cmp_.apply(top_frame, axis=1)
print(len(cmp_), "articles;", int(cmp_["dominant_frame"].notna().sum()), "have a dataset dominant_frame;", int(cmp_[f"confirmed_{FRAMES[0]}"].isna().sum()), "not found in the dataset by URL")

420 articles; 304 have a dataset dominant_frame; 0 not found in the dataset by URL


In [8]:
# (a) per frame: ours (>= 1 hit) vs dataset confirmed
rows = []
for f in FRAMES:
    ours = (cmp_[f"hits_{f}"] >= 1).astype(int).to_numpy(); theirs = cmp_[f"confirmed_{f}"].to_numpy()
    tp = int(((ours == 1) & (theirs == 1)).sum())
    rows.append({"frame": f, "dataset_confirmed_%": round(100 * theirs.mean(), 1), "ours_present_%": round(100 * ours.mean(), 1),
                 "agreement_%": round(100 * (ours == theirs).mean(), 1), "kappa": round(cohen_kappa(theirs, ours, [0, 1]), 3),
                 "precision_vs_dataset": round(tp / max(ours.sum(), 1), 3), "recall_vs_dataset": round(tp / max(theirs.sum(), 1), 3)})
per_frame = pd.DataFrame(rows); per_frame.to_csv(OUT / "dataset_label_comparison_per_frame.csv", index=False)
per_frame

,frame,dataset_confirmed_%,ours_present_%,agreement_%,kappa,precision_vs_dataset,recall_vs_dataset
0,innovation_opportunity,8.8,54.3,46.9,0.008,0.092,0.568
1,risk_safety,13.1,61.9,47.9,0.113,0.185,0.873
2,regulation_governance,7.9,54.3,47.9,0.027,0.092,0.636
3,rights_privacy,6.7,31.4,68.6,0.073,0.106,0.500
4,economy_labour,6.9,61.0,43.1,0.043,0.090,0.793
5,misinformation_integrity,2.6,25.0,75.2,0.059,0.057,0.545


In [9]:
# (b) our top frame vs the dataset dominant_frame (articles where the dataset has one)
both = cmp_[cmp_["dominant_frame"].notna()]
rep_d = agreement_report(both["dominant_frame"], both["our_top_frame"], LABELS)
print("articles compared:", rep_d["n"], "| exact agreement:", round(rep_d["accuracy"], 3), "| kappa:", round(rep_d["kappa"], 3))
rep_d["confusion"].to_csv(OUT / "dataset_label_comparison_confusion.csv"); rep_d["per_class"].round(3).to_csv(OUT / "dataset_label_comparison_dominant_frame.csv", index=False)
display(rep_d["confusion"])
rep_d["per_class"].round(3)

articles compared: 304 | exact agreement: 0.352 | kappa: 0.197


checked,innovation_opportunity,risk_safety,regulation_governance,rights_privacy,economy_labour,misinformation_integrity,none
reference,,,,,,,
innovation_opportunity,14,11,12,1,11,2,0
risk_safety,4,37,7,6,7,0,0
regulation_governance,14,37,20,8,19,7,1
rights_privacy,5,10,5,11,1,1,2
economy_labour,6,6,7,0,24,1,0
misinformation_integrity,1,2,1,1,0,1,1
none,0,0,0,0,0,0,0


,label,n_reference,n_checked,true_positive,precision,recall,f1
0,innovation_opportunity,51,44,14,0.318,0.275,0.295
1,risk_safety,61,103,37,0.359,0.607,0.451
2,regulation_governance,106,52,20,0.385,0.189,0.253
3,rights_privacy,35,27,11,0.407,0.314,0.355
4,economy_labour,44,62,24,0.387,0.545,0.453
5,misinformation_integrity,7,12,1,0.083,0.143,0.105
6,none,0,4,0,0.000,NaN,NaN


In [10]:
# examples where they disagree (ids, titles and counts only; no article text)
dis = both[both["dominant_frame"] != both["our_top_frame"]].copy()
dis["our_hits_in_dataset_frame"] = [r[f"hits_{r['dominant_frame']}"] for _, r in dis.iterrows()]
ex = dis.sample(min(25, len(dis)), random_state=SEED)[["doc_id", "region", "dominant_frame", "our_top_frame", "hits_total", "our_hits_in_dataset_frame", "title", "url"]]
ex["title"] = ex["title"].str[:80]
ex.to_csv(OUT / "dataset_label_disagreement_examples.csv", index=False)
print(len(dis), "of", len(both), "articles disagree on the top frame; 25 examples saved to dataset/analysis/dataset_label_disagreement_examples.csv")
ex.head(10)

197 of 304 articles disagree on the top frame; 25 examples saved to dataset/analysis/dataset_label_disagreement_examples.csv


,doc_id,region,dominant_frame,our_top_frame,hits_total,our_hits_in_dataset_frame,title,url
300,news_0301,EU,innovation_opportunity,regulation_governance,3,0,Microsoft Scout Is the New AI Assistant That N...,https://memeburn.com/microsoft-scout-is-the-ne...
249,news_0250,US,rights_privacy,innovation_opportunity,2,0,"The AI Ethics Waterfall: Disclosure, Governanc...",https://ipwatchdog.com/2026/03/09/ai-ethics-wa...
27,news_0028,US,regulation_governance,innovation_opportunity,5,1,Agentic AI Emerges as the Next Frontier for St...,https://www.bankinfosecurity.com/agentic-ai-em...
175,news_0176,US,regulation_governance,innovation_opportunity,9,0,Pentagon casts cloud of doubt over Anthropic’s...,https://telecom.economictimes.indiatimes.com/n...
342,news_0343,UK,innovation_opportunity,economy_labour,8,0,AI might not be coming for lawyers’ jobs anyti...,https://www.technologyreview.com/2025/12/15/11...
391,news_0392,EU,risk_safety,regulation_governance,10,2,Grok’s antisemitic outburst heaps pressure on ...,https://www.politico.eu/article/grok-artificia...
163,news_0164,Other,innovation_opportunity,risk_safety,2,0,Google API Keys Weren't Secrets. But then Gemi...,https://trufflesecurity.com/blog/google-api-ke...
120,news_0121,UK,regulation_governance,economy_labour,7,2,AI salaries in UAE compete with US but a skill...,https://www.thenationalnews.com/future/technol...
29,news_0030,Other,economy_labour,risk_safety,16,1,Pentagon's Use of Grok Raises AI Security Conc...,https://www.govinfosecurity.com/pentagons-use-...
363,news_0364,EU,rights_privacy,risk_safety,3,1,WINDOWS FORUM: AI Amplified Bot War: Shell vs ...,https://royaldutchshellplc.com/2026/03/04/wind...


## Limits to state in the report
- The dataset's labels come from GDELT entity names and quotations, ours from article text, so the comparison is between two noisy instruments; see the confusion matrix for where they differ.
- The dataset labels only 41% of articles (null = unconfirmed), and its governance base rate is inflated by the corpus's own governance filter (dataset card), so agreement on `regulation_governance` is partly built in.
- Human validation (section 1) is the real test of the dictionary; this comparison is a cross-check, not a replacement.